# Chunk header run: does listing merged definitions help?

1. Upload **phase4_header_source.zip** to the root of My Drive.
2. Select an **A100 GPU** in Colab and Run all. Approve Drive mounting.
3. Bring the downloaded **phase4_header_results.zip** back for analysis.

Production chunks that merge several small definitions are headed only by their path. The
variant adds one line, `# defines: a, b, c`, to those chunks (`Settings.header_defines`), which
changes the text that is embedded, indexed by BM25 and shown to the reranker.

For each repo the notebook indexes the pinned code twice (header off, then on) and runs two
policies on the same 120 cases as the policy run: **hybrid** (no reranking) and **plain**
(rerank the fused top 20 with the single code instruction). Nothing is tuned. BM25-only on CPU was
neutral to slightly negative (MRR 0.478 to 0.472); this measures dense, fusion and reranking.

Quality only; labels are author-checked, not independently reviewed. A100 is for experiments.
Expect roughly 15 to 30 minutes (vectors for the changed header are embedded again).

In [ ]:
SOURCE_ZIP = "/content/drive/MyDrive/phase4_header_source.zip"

In [ ]:
# Load the separate source ZIP that you uploaded to Google Drive.
import hashlib, pathlib, zipfile
from google.colab import drive

drive.mount("/content/drive")
source_zip = pathlib.Path(SOURCE_ZIP)
if not source_zip.is_file():
    raise FileNotFoundError(
        f"Upload phase4_header_source.zip to My Drive, or update SOURCE_ZIP. Missing: {source_zip}"
    )
expected_sha256 = "1cfff217f9ff02c0c58aae722b26bdfdca7a73685201f9237101ae99f6fbbec0"
if hashlib.sha256(source_zip.read_bytes()).hexdigest() != expected_sha256:
    raise ValueError(
        "Source ZIP does not match this notebook. Download the matching ZIP and notebook again."
    )
PAYLOAD = pathlib.Path("/content/phase4_header_payload")
with zipfile.ZipFile(source_zip) as package:
    package.extractall(PAYLOAD)
print("Loaded project code and question datasets from", source_zip.name)

In [ ]:
import importlib.util, subprocess, sys
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], check=True)
if importlib.util.find_spec('uv') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--system',
                '--python', sys.executable, '-q', 'chromadb==1.5.9', 'fastapi==0.162.0', 'gitpython==3.2.1', 'langchain-chroma==1.1.0', 'langchain-core==1.6.13', 'langchain-text-splitters==1.1.3', 'langgraph==1.2.19', 'pydantic==2.14.1', 'pydantic-settings==2.16.1', 'rank-bm25==0.2.2', 'rich==15.0.0', 'sentence-transformers==5.7.0', 'transformers==4.57.6', 'tree-sitter==0.26.0', 'tree-sitter-language-pack==1.22.1', 'typer==0.27.3', 'uvicorn==0.54.0', 'astchunk==0.1.0'], check=True)

In [ ]:
import json, os, pathlib, shutil, subprocess, sys
REPOS = pathlib.Path('/content/header_repos')
RESULTS = pathlib.Path('/content/header_results')
LABELS = PAYLOAD / 'eval/datasets/phase4_intent_v1'
if RESULTS.exists():
    raise FileExistsError('Results already exist. Download them before starting a new runtime.')
REPOS.mkdir(exist_ok=True)
RESULTS.mkdir()
shutil.copytree(LABELS, RESULTS / 'labels')
shutil.copyfile(source_zip, RESULTS / 'phase4_header_source.zip')
env = {**os.environ, 'PYTHONPATH': os.pathsep.join([str(PAYLOAD), str(PAYLOAD / 'src')]),
       'TOKENIZERS_PARALLELISM': 'false'}
manifest = json.loads((LABELS / 'manifest.json').read_text())
for entry in manifest['datasets']:
    repo = entry['repo']
    checkout = REPOS / repo.replace('/', '__')
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout',
                        'https://github.com/' + repo, str(checkout)], check=True)
    subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach',
                    entry['commit_sha']], check=True)
    dataset = LABELS / entry['file']
    if hashlib.sha256(dataset.read_bytes()).hexdigest() != entry['sha256']:
        raise ValueError('Dataset checksum mismatch')
    for variant, extra in (('header_off', []), ('header_on', ['--header-defines'])):
        subprocess.run([sys.executable, str(PAYLOAD / 'eval/run_policy_eval.py'),
                        '--dataset', str(dataset), '--checkout', str(checkout),
                        '--output', str(RESULTS / (repo.replace('/', '__') + '__' + variant)),
                        '--protocol', str(LABELS / 'protocol.json'),
                        '--policies', 'hybrid,plain', *extra,
                        '--mode', 'full', '--device', 'cuda'], env=env, check=True)
for repo_dir in sorted(RESULTS.iterdir()):
    summary = repo_dir / 'summary.md'
    if summary.is_file():
        print(repo_dir.name)
        print(summary.read_text())

In [ ]:
from google.colab import files
archive = pathlib.Path('/content/phase4_header_results.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(RESULTS.rglob('*')):
        if path.is_file() and 'isolated_index' not in path.parts:
            bundle.write(path, path.relative_to(RESULTS))
files.download(str(archive))